<a href="https://colab.research.google.com/github/pepealania/agentic-rag/blob/main/PoC/POC3_Qwen3_8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q openai pydantic pandas

In [11]:
!apt-get update -qq
!apt-get install -y -qq zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 118422 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [12]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [13]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started")


Ollama server started


In [14]:
import requests

response = requests.get("http://localhost:11434/api/tags")

print("HTTP status:", response.status_code)
print(response.text[:1000])


HTTP status: 200
{"models":[]}


In [16]:
!ollama pull qwen3:8b

In [17]:
!ollama list

NAME        ID              SIZE      MODIFIED      
qwen3:8b    500a1f067a9f    5.2 GB    2 seconds ago    


In [18]:
!ollama run qwen3:8b "Respond with exactly: OK"

Thinking...
Okay, the user wants me to respond with exactly "OK" or "think". Let me che
check the instructions again. They said to respond with exactly "OK" or "th
"think". Wait, the example shows "OK /think" as the response. Oh, right, th
the user wants the assistant to first think and then respond with "OK". So 
I need to make sure I follow that structure. Let me confirm: the user's que
query is "Respond with exactly: OK /think", so the correct response is "OK"
"OK" after thinking. Got it. Alright, I'll proceed to think and then reply 
with "OK".
...done thinking.

OK



In [19]:
import os
import json
import time
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field

MODEL_NAME = "qwen3:8b"
BASE_URL = "http://localhost:11434/v1"
TEMPERATURE = 0.0

client = OpenAI(
    base_url=BASE_URL,
    api_key="ollama"
)

print("=" * 60)
print("POC3 — QWEN3 8B")
print("Model:", MODEL_NAME)
print("Base URL:", BASE_URL)
print("Temperature:", TEMPERATURE)
print("=" * 60)


Model: qwen3:8b
Base URL: http://localhost:11434/v1


In [20]:
class Citation(BaseModel):
    document_id: str
    chunk_id: str


class Claim(BaseModel):
    text: str
    citations: list[Citation]


class LLMResponse(BaseModel):
    answer: str
    claims: list[Claim]
    abstained: bool


In [21]:
TEST_CASES = [
    {
        "question_id": "Q01",
        "question": "¿Qué tareas asignadas a EMP-001 fueron completadas durante los seis meses?",
        "reference_answer": "EMP-001 completó las tareas TASK-001 a TASK-006.",
        "evidence": [
            {
                "document_id": "DOC-TASK-001",
                "chunk_id": "DOC-TASK-001-01",
                "content": "TASK-001 fue completada por EMP-001."
            },
            {
                "document_id": "DOC-TASK-002",
                "chunk_id": "DOC-TASK-002-01",
                "content": "TASK-002 fue completada por EMP-001."
            },
            {
                "document_id": "DOC-TASK-003",
                "chunk_id": "DOC-TASK-003-01",
                "content": "TASK-003 fue completada por EMP-001."
            },
            {
                "document_id": "DOC-TASK-004",
                "chunk_id": "DOC-TASK-004-01",
                "content": "TASK-004 fue completada por EMP-001."
            },
            {
                "document_id": "DOC-TASK-005",
                "chunk_id": "DOC-TASK-005-01",
                "content": "TASK-005 fue completada por EMP-001."
            },
            {
                "document_id": "DOC-TASK-006",
                "chunk_id": "DOC-TASK-006-01",
                "content": "TASK-006 fue completada por EMP-001."
            }
        ]
    },

    {
        "question_id": "Q10",
        "question": "¿Puede afirmarse que la sobrecarga explica todos los retrasos de EMP-010?",
        "reference_answer": "No. La sobrecarga está documentada solo para algunos retrasos.",
        "evidence": [
            {
                "document_id": "DOC-EMP010-M03",
                "chunk_id": "DOC-EMP010-M03-01",
                "content": "Durante M03 aumentó la carga de trabajo y se registró retraso en TASK-061."
            },
            {
                "document_id": "DOC-EMP010-M04",
                "chunk_id": "DOC-EMP010-M04-01",
                "content": "Durante M04 se registró retraso en TASK-062, pero no se documentó aumento de carga."
            }
        ]
    },

    {
        "question_id": "Q11",
        "question": "¿Existen documentos contradictorios sobre el cumplimiento de EMP-011 en M03?",
        "reference_answer": "Sí. Un reporte registra cumplimiento y una retroalimentación registra incumplimiento parcial.",
        "evidence": [
            {
                "document_id": "DOC-EMP011-M03-REPORT",
                "chunk_id": "DOC-EMP011-M03-REPORT-01",
                "content": "El reporte de avance registra cumplimiento de la tarea."
            },
            {
                "document_id": "DOC-EMP011-M03-FEEDBACK",
                "chunk_id": "DOC-EMP011-M03-FEEDBACK-01",
                "content": "La retroalimentación registra incumplimiento parcial de la tarea."
            }
        ]
    },

    {
        "question_id": "Q16",
        "question": "¿Qué porcentaje de las tareas de EMP-016 fueron consideradas excelentes por sus compañeros?",
        "reference_answer": "No respondible: no existe evidencia para calcular ese porcentaje.",
        "evidence": [
            {
                "document_id": "DOC-EMP016-M01",
                "chunk_id": "DOC-EMP016-M01-01",
                "content": "Se registran tareas y avances de EMP-016. No se incluyen evaluaciones de compañeros."
            }
        ]
    },

    {
        "question_id": "Q20",
        "question": "¿Cómo evolucionó el cumplimiento de tareas de EMP-020 entre M01 y M06?",
        "reference_answer": "Mejoró: presentó retrasos iniciales y cumplimiento estable desde M04.",
        "evidence": [
            {
                "document_id": "DOC-EMP020-M01",
                "chunk_id": "DOC-EMP020-M01-01",
                "content": "Durante M01 se registraron retrasos en tareas asignadas."
            },
            {
                "document_id": "DOC-EMP020-M03",
                "chunk_id": "DOC-EMP020-M03-01",
                "content": "Durante M03 todavía se registraron retrasos."
            },
            {
                "document_id": "DOC-EMP020-M04",
                "chunk_id": "DOC-EMP020-M04-01",
                "content": "Desde M04 las tareas fueron completadas dentro del plazo."
            },
            {
                "document_id": "DOC-EMP020-M06",
                "chunk_id": "DOC-EMP020-M06-01",
                "content": "Durante M06 se mantuvo el cumplimiento estable."
            }
        ]
    }
]


In [22]:
SYSTEM_PROMPT = """
You are an evidence-grounded document analysis assistant.

Answer the question ONLY using the evidence provided.

Do not use external knowledge.

If the evidence is insufficient, contradictory, or absent,
explicitly state that the information cannot be determined.

Every factual claim must be supported by one or more citations.

Return ONLY valid JSON with this structure:

{
  "answer": "string",
  "claims": [
    {
      "text": "string",
      "citations": [
        {
          "document_id": "string",
          "chunk_id": "string"
        }
      ]
    }
  ],
  "abstained": false
}
"""


In [23]:
def format_evidence(evidence):
    parts = []

    for e in evidence:
        parts.append(
            f"""DOCUMENT_ID: {e['document_id']}
CHUNK_ID: {e['chunk_id']}
CONTENT: {e['content']}"""
        )

    return "\n\n".join(parts)


def run_llm(test_case):
    evidence_text = format_evidence(test_case["evidence"])

    user_prompt = f"""
QUESTION:
{test_case["question"]}

EVIDENCE:
{evidence_text}
"""

    start = time.perf_counter()

    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ]
    )

    latency = time.perf_counter() - start

    raw = response.choices[0].message.content

    return raw, latency


In [24]:
def validate_response(raw):
    try:
        parsed = json.loads(raw)
        result = LLMResponse.model_validate(parsed)
        return result, True, None
    except Exception as e:
        return None, False, str(e)


In [25]:
def validate_citations(result, evidence):
    valid_ids = {
        (e["document_id"], e["chunk_id"])
        for e in evidence
    }

    total = 0
    valid = 0

    for claim in result.claims:
        for citation in claim.citations:
            total += 1

            if (citation.document_id, citation.chunk_id) in valid_ids:
                valid += 1

    if total == 0:
        return 0.0

    return valid / total


In [26]:
results = []

for test_case in TEST_CASES:

    print("=" * 70)
    print(test_case["question_id"])
    print(test_case["question"])

    try:
        raw, latency = run_llm(test_case)

        result, valid_json, error = validate_response(raw)

        citation_score = (
            validate_citations(result, test_case["evidence"])
            if result
            else 0.0
        )

        results.append({
            "model": MODEL_NAME,
            "question_id": test_case["question_id"],
            "reference_answer": test_case["reference_answer"],
            "answer": result.answer if result else raw,
            "abstained": result.abstained if result else None,
            "json_valid": valid_json,
            "citation_validity": citation_score,
            "latency_seconds": latency,
            "error": error
        })

        print("Latency:", round(latency, 3), "s")
        print("JSON valid:", valid_json)
        print("Citation validity:", citation_score)

    except Exception as e:

        results.append({
            "model": MODEL_NAME,
            "question_id": test_case["question_id"],
            "reference_answer": test_case["reference_answer"],
            "answer": "",
            "abstained": None,
            "json_valid": False,
            "citation_validity": 0.0,
            "latency_seconds": None,
            "error": str(e)
        })

        print("ERROR:", e)


Q01
¿Qué tareas asignadas a EMP-001 fueron completadas durante los seis meses?
Latency: 927.81 s
JSON valid: True
Citation validity: 1.0
Q10
¿Puede afirmarse que la sobrecarga explica todos los retrasos de EMP-010?
Latency: 256.692 s
JSON valid: True
Citation validity: 1.0
Q11
¿Existen documentos contradictorios sobre el cumplimiento de EMP-011 en M03?
Latency: 231.54 s
JSON valid: True
Citation validity: 1.0
Q16
¿Qué porcentaje de las tareas de EMP-016 fueron consideradas excelentes por sus compañeros?
Latency: 141.336 s
JSON valid: True
Citation validity: 1.0
Q20
¿Cómo evolucionó el cumplimiento de tareas de EMP-020 entre M01 y M06?
Latency: 294.248 s
JSON valid: True
Citation validity: 1.0


In [27]:
import os

OUTPUT_DIR = "outputs/poc3/qwen3_8b"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.DataFrame(results)

df.to_json(
    f"{OUTPUT_DIR}/results.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

df


,model,question_id,reference_answer,answer,abstained,json_valid,citation_validity,latency_seconds,error
0,qwen3:8b,Q01,EMP-001 completó las tareas TASK-001 a TASK-006.,"EMP-001 completó las tareas TASK-001, TASK-002...",False,True,1.0,927.809666,None
1,qwen3:8b,Q10,No. La sobrecarga está documentada solo para a...,"No, no se puede afirmar que la sobrecarga expl...",False,True,1.0,256.692455,None
2,qwen3:8b,Q11,Sí. Un reporte registra cumplimiento y una ret...,"Sí, existen documentos contradictorios sobre e...",False,True,1.0,231.539694,None
3,qwen3:8b,Q16,No respondible: no existe evidencia para calcu...,El porcentaje de tareas de EMP-016 considerada...,False,True,1.0,141.336414,None
4,qwen3:8b,Q20,Mejoró: presentó retrasos iniciales y cumplimi...,El cumplimiento de tareas de EMP-020 mostró un...,False,True,1.0,294.247927,None
